# Lunar Myth Kintsugi Botanist — LoRA Training Notebook

**By Jennipher Troup**

This Colab notebook trains a custom SDXL LoRA for the Lunar Myth Kintsugi style.

**Requirements:**
- Google account with Google Drive
- 20–100 curated training images in `MyDrive/LunarMyth_Dataset/dataset/`
- Each image needs a matching `.txt` caption file

**Recommended GPU:** A100 or T4 (Colab free tier works, slower)

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted successfully")

## Step 2: Install Dependencies

In [ ]:
!pip install -q torch==2.1.2+cu121 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers==0.0.23.post1
!git clone https://github.com/bmaltais/kohya_ss.git /content/kohya_ss
%cd /content/kohya_ss
!pip install -q -r requirements.txt
!pip install -q prodigy-opt
print("Dependencies installed")

## Step 3: Verify Installation

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## Step 4: Configure Training Parameters

In [ ]:
# === CONFIGURATION === Edit these paths and values as needed

DATASET_DIR = "/content/drive/MyDrive/LunarMyth_Dataset/dataset"
OUTPUT_DIR = "/content/drive/MyDrive/LunarMyth_Output"
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
OUTPUT_NAME = "lunar_myth_kintsugi_botanist_v1"

# Network architecture
NETWORK_DIM = 48      # Rank: 32-64 for complex styles
NETWORK_ALPHA = 24    # Typically half of dim

# Training length
MAX_TRAIN_STEPS = 1200
SAVE_EVERY_N_STEPS = 200

# Training efficiency
RESOLUTION = "1024,1024"
TRAIN_BATCH_SIZE = 1  # Increase to 2 if 16GB+ VRAM

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Model name: {OUTPUT_NAME}")

## Step 5: Launch Training (Prodigy Optimizer)

In [ ]:
!accelerate launch \
  --mixed_precision="bf16" \
  --num_processes=1 \
  --num_machines=1 \
  /content/kohya_ss/sdxl_train_network.py \
  --pretrained_model_name_or_path="{BASE_MODEL}" \
  --train_data_dir="{DATASET_DIR}" \
  --output_dir="{OUTPUT_DIR}" \
  --output_name="{OUTPUT_NAME}" \
  --network_module="networks.lora" \
  --network_dim={NETWORK_DIM} \
  --network_alpha={NETWORK_ALPHA} \
  --max_train_steps={MAX_TRAIN_STEPS} \
  --save_every_n_steps={SAVE_EVERY_N_STEPS} \
  --learning_rate=1.0 \
  --optimizer_type="Prodigy" \
  --optimizer_args="decouple=True weight_decay=0.01 d0=0.0001 use_bias_correction=True safeguard_warmup=True betas=0.9,0.99 d_coef=2.0" \
  --lr_scheduler="constant" \
  --resolution="{RESOLUTION}" \
  --train_batch_size={TRAIN_BATCH_SIZE} \
  --mixed_precision="bf16" \
  --save_precision="bf16" \
  --cache_latents \
  --cache_latents_to_disk \
  --gradient_checkpointing \
  --caption_extension=".txt" \
  --shuffle_caption \
  --keep_tokens=1 \
  --enable_bucket \
  --min_bucket_reso=640 \
  --max_bucket_reso=1536 \
  --save_model_as="safetensors" \
  --clip_skip=2

## Step 6: Verify Output

In [ ]:
import os
output_files = sorted(os.listdir(OUTPUT_DIR))
print("Generated files:")
for f in output_files:
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024*1024)
    print(f"  {f} ({size_mb:.1f} MB)")

## Step 7: Test Your LoRA

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe = pipe.to("cuda")

# Load your trained LoRA
lora_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}.safetensors")
pipe.load_lora_weights(lora_path)

prompt = "lunar myth kintsugi botanist full-body female primordial goddess, luminous wonder mood, newspaper collage with watercolor bleed, stained-glass mosaic and obsidian with golden kintsugi, regenerative moonflowers and iridescent ivy, heavy embroidery and beading, datamosh glitch, lunarpunk moonlight, vibrant jewel tones, chromatic aberration"

image = pipe(
    prompt,
    num_inference_steps=40,
    guidance_scale=7.5,
    height=1216,
    width=832
).images[0]

image.save("/content/test_lora_output.png")
print("Test image saved to /content/test_lora_output.png")
display(image)

---

## Troubleshooting

| Issue | Solution |
|-------|----------|
| Out of memory | Reduce batch_size to 1, enable gradient_checkpointing |
| Loss oscillating | Lower `d0` to `0.00005`, reduce `d_coef` to `1.0` |
| Style weak | Increase steps to 1500+, increase network_dim to 64 |
| Faces same | Add more diverse faces, use lower network_dim (32) |
| Colors muddy | Lower learning rate, reduce `d0` |

**Break. Repair. Glow. Repeat.**